# Trening TrOCR: sprawdzone transkrypcje

Wybierz **GPU T4** i **Uruchom wszystko**. Nie wgrywaj zadnych plikow.

70 sprawdzonych linii treningowych, 500 probek zwyklego druku, 3 epoki LoRA. Kontrola: 9 linii historycznych z oddzielnej kolekcji i 75 zwyklych linii. Ryzykowne rodziny dziel sa wykluczone. To eksperyment, nie niezalezny benchmark SOTA. Oryginalne teksty i stare glify pozostaja bez modernizacji.

Po treningu pobierze sie jeden ZIP z wagami i raportem.

## 1. Srodowisko

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import venv

CODE_REVISION = 'fe074fee1da33a94ea60a0bcd658a8f6afbb7dc8'
DATA_REVISION = '3a301fd12a6272b28774d1888213ddf528b47f24'
DATA_SHA256 = '3e39b7d02b90a0a741adde506c0467698c46fe86c82846d3cc0e648eebaae32e'
repo = Path('/content/OCR_engine-reviewed')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
environment = Path('/content/slayer-reviewed-training-env')
python = environment/'bin/python'
if not python.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
subprocess.run([str(python), '-c', "import torch; assert torch.cuda.is_available(), 'Wybierz GPU T4 w ustawieniach Colaba i uruchom wszystko.'; print('GPU:', torch.cuda.get_device_name(0))"], check=True)
torch_release = subprocess.check_output([str(python), '-c', "import torch; print('.'.join(torch.__version__.split('+')[0].split('.')[:2]))"], text=True).strip()
torchao_versions = {'2.10': '0.16.0', '2.11': '0.17.0'}
assert torch_release in torchao_versions, f'Niezweryfikowana wersja Torch: {torch_release}. Nie zmieniam CUDA.'
packages = ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2',
            'peft==0.19.1', 'accelerate==1.13.0', 'jiwer==4.0.0',
            'pillow==11.3.0', 'sentencepiece==0.2.1']
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir', *packages], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir', '--no-deps',
                'torchao=='+torchao_versions[torch_release]], check=True)
subprocess.run([str(python), '-c', "from transformers import TrOCRProcessor, VisionEncoderDecoderModel; from transformers.generation import GenerationMixin; import peft, jiwer; print('TRAINING_IMPORTS_OK')"], check=True)
subprocess.run([str(python), '-m', 'training.preflight_reviewed_recognizer',
                '--output', str(environment/'adapter-preflight.json')], cwd=repo, check=True)
print('Gotowe. Dane zostana pobrane automatycznie; niczego nie wgrywaj.')


## 2. Dane, kontrola, trening i porownanie

In [ ]:
from google.colab import files

RESULT_DIRECTORY = None
EVIDENCE_ZIP = None
command = [str(python), '-m', 'training.run_reviewed_recognizer_colab',
           '--dataset-revision', DATA_REVISION, '--dataset-sha256', DATA_SHA256]
process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
    if line.startswith('RESULT_DIRECTORY '):
        RESULT_DIRECTORY = Path(line.strip().split(' ', 1)[1])
    elif line.startswith('EVIDENCE_ZIP '):
        EVIDENCE_ZIP = Path(line.strip().split(' ', 1)[1])
return_code = process.wait()
if return_code:
    if EVIDENCE_ZIP is not None and EVIDENCE_ZIP.is_file():
        files.download(str(EVIDENCE_ZIP))
    raise RuntimeError('Trening nie zakonczyl sie. Zapisano ZIP diagnostyczny; nie jest to model.')
assert RESULT_DIRECTORY is not None, 'Brak potwierdzenia zakonczenia treningu.'
print('TRENING_ZAKONCZONY', RESULT_DIRECTORY)


## 3. Pobranie modelu i raportu

In [ ]:
assert RESULT_DIRECTORY is not None, 'Najpierw musi zakonczyc sie komorka treningu.'
result_zip = RESULT_DIRECTORY/'recognizer-reviewed-colab-v1-result.zip'
assert result_zip.is_file(), 'Brak kompletnego archiwum wyniku.'
print('Model i raport w jednym pliku:', result_zip)
files.download(str(result_zip))
